# Integrating Data with Merge, Join, and Concat

Combine tables without accidentally duplicating records or losing unmatched keys.

> 🧭 This notebook focuses on practical data work. Examples are intentionally compact and do not cover machine learning.

## Learning goals
- Choose between `merge`, `join`, and `concat`.
- Use merge validation and indicator columns.
- Inspect unmatched records and normalize join keys.
- Detect one-to-many joins that inflate counts or totals.

In [1]:
import pandas as pd

orders = pd.DataFrame({"order_id": [1, 2, 3, 4], "customer_code": [" c1 ", "C2", "C3", "C9"], "amount": [50, 20, 35, 15]})
customers = pd.DataFrame({"customer_code": ["C1", "C2", "C3"], "segment": ["Retail", "Business", "Retail"]})
orders["customer_code"] = orders["customer_code"].str.strip().str.upper()
customers["customer_code"] = customers["customer_code"].str.strip().str.upper()
display(orders)
display(customers)

,order_id,customer_code,amount
0,1,C1,50
1,2,C2,20
2,3,C3,35
3,4,C9,15


,customer_code,segment
0,C1,Retail
1,C2,Business
2,C3,Retail


## Merge type and diagnostics
- `inner`: keep only keys matched on both sides.
- `left`: keep every left row and attach matches where present.
- `outer`: keep all keys from both sides.
- `validate="many_to_one"` asserts that the right-hand key is unique.
- `indicator=True` labels each output row as left-only, right-only, or matched.

In [2]:
merged = orders.merge(customers, on="customer_code", how="left", validate="many_to_one", indicator=True)
display(merged)
print(merged["_merge"].value_counts())
print("Order total before:", orders["amount"].sum())
print("Order total after:", merged["amount"].sum())

_merge
both          3
left_only     1
right_only    0
Name: count, dtype: int64
Order total before: 120
Order total after: 120


,order_id,customer_code,amount,segment,_merge
0,1,C1,50,Retail,both
1,2,C2,20,Business,both
2,3,C3,35,Retail,both
3,4,C9,15,NaN,left_only


## Why join keys matter
If the lookup table contains multiple rows for a key, joining can multiply rows. Resolve duplicates or define a proper composite key before merging.

In [3]:
bad_lookup = pd.DataFrame({"customer_code": ["C1", "C1", "C2"], "manager": ["Ana", "Ben", "Cem"]})
try:
    orders.merge(bad_lookup, on="customer_code", validate="many_to_one")
except pd.errors.MergeError as error:
    print("Validation caught the problem:", error)

# Concatenate compatible tables vertically.
jan = pd.DataFrame({"month": ["Jan"], "sales": [120]})
feb = pd.DataFrame({"month": ["Feb"], "sales": [135]})
display(pd.concat([jan, feb], ignore_index=True))

Validation caught the problem: Merge keys are not unique in right dataset; not a many-to-one merge


,month,sales
0,Jan,120
1,Feb,135


## Practice
Merge a table of store IDs with a table of store names. Use a left join and report stores without a matching name.

In [4]:
store_sales = pd.DataFrame({"store_id": [10, 11, 12], "sales": [200, 150, 180]})
store_names = pd.DataFrame({"store_id": [10, 11], "store_name": ["North", "Central"]})
answer = store_sales.merge(store_names, how="left", on="store_id", indicator=True, validate="many_to_one")
display(answer)
print("Unmatched store IDs:", answer.loc[answer["_merge"] == "left_only", "store_id"].tolist())

Unmatched store IDs: [12]


,store_id,sales,store_name,_merge
0,10,200,North,both
1,11,150,Central,both
2,12,180,NaN,left_only
